# Diet-Derived Chemical Exposure And Mental Health

Exploratory TRE-side notebook for testing whether participants with higher estimated dietary exposure to selected food chemicals differ in mental-health survey metrics.

This is intended as a side-paper/application analysis for Diet Data Enhancement. It is **not causal** by itself. It estimates exposure from logged diet and food-level/KG annotations, assigns participants to low/mid/high exposure groups, and compares mental-health phenotypes across groups.

Default examples:

- caffeine
- lead
- acrylamide

The current de novo enriched tables clearly contain caffeine. Lead/acrylamide may be absent unless they are present in the uploaded KG/reference tables; the notebook reports that explicitly.

In [ ]:
from __future__ import annotations

import json
import math
import re
import sys
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")


def find_project_root(start: Path | None = None) -> Path:
    """Find the Diet Data Enhancement project root from any notebook cwd."""
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    strong_markers = [
        Path("outputs/downstream_features"),
        Path("outputs/enhanced_hpp"),
        Path("outputs/visualizations/denovo_hpp_kg_mega_flexible_data.js"),
    ]
    for candidate in candidates:
        if (candidate / "downstream_analysis").is_dir() and any((candidate / marker).exists() for marker in strong_markers):
            return candidate
    for candidate in candidates:
        if (candidate / "downstream_analysis").is_dir() and (candidate / "outputs").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find project root. Start Jupyter from Diet_Data_Enhancement_TRE "
        "or set PROJECT_ROOT manually to that folder."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / "downstream_analysis" / "manual" / "outputs" / "diet_mental_health"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT, OUTPUT_DIR


## Configuration

Change these values for the analysis you want to run. `EXPOSURE_SPECS` controls the food chemical/exposure definitions. `MENTAL_HEALTH_DATASET` and `MENTAL_HEALTH_TABLE` follow HPP `PhenoLoader` names.

In [ ]:
DIET_EVENTS_PATH = PROJECT_ROOT / "tre_inputs" / "diet_logging_events.parquet"
FOOD_FEATURE_TABLE = PROJECT_ROOT / "outputs" / "downstream_features" / "denovo" / "mental_health" / "hpp_downstream_feature_table.csv"

# KG_MODE controls how exposure discovery works.
# "full_mega" uses the full denovo mega KG payload used by the interactive mega visualization.
# "scenario" uses the smaller HPP scenario KG CSVs.
KG_MODE = "full_mega"  # full_mega, scenario
MEGA_KG_DATA_PATH = PROJECT_ROOT / "outputs" / "visualizations" / "denovo_hpp_kg_mega_flexible_data.js"
SCENARIO_KG_EDGES_PATH = PROJECT_ROOT / "outputs" / "enhanced_hpp" / "1.denovo" / "kg" / "hpp_scenario_kg_edges.csv"
SCENARIO_KG_NODES_PATH = PROJECT_ROOT / "outputs" / "enhanced_hpp" / "1.denovo" / "kg" / "hpp_scenario_kg_nodes.csv"

MENTAL_HEALTH_DATASET = "psychological_and_social_health"
MENTAL_HEALTH_TABLE = "psychological_and_social_health"
MENTAL_HEALTH_LOCAL_PATH = PROJECT_ROOT / "tre_inputs" / "psychological_and_social_health.parquet"

ID_COL = "participant_id"
FOOD_COL = "food_id"
REF_FOOD_COL = "hpp_food_id"
GRAMS_COL = "weight_g"

# Use explicit names confirmed from the mega KG / feature table.
# Avoid abbreviations or generated variants unless they are exact KG labels.
EXPOSURE_SPECS = {
    "caffeine": {
        "terms": ["caffeine"],
        "preferred_feature_columns": ["Caffeine", "kg__has_nutrient_amount_per_100g__nutrient__caffeine"],
        "source_note": "Numeric nutrient/KG feature if available; otherwise KG connected-food grams.",
    },
    "lead": {
        "terms": ["lead"],
        "preferred_feature_columns": ["Lead"],
        "source_note": "Likely absent unless heavy-metal annotations are in the KG/reference table.",
    },
    "acrylamides": {
        "terms": ["acrylamides"],
        "preferred_feature_columns": ["Acrylamides"],
        "source_note": "FoodAtlas full-KG label; scenario KG may not contain this chemical.",
    },
}

DEFAULT_OUTCOME_PATTERNS = [
    "depress", "disinterest", "nervous", "worrier", "worry", "tense", "restless", "fed_up", "tired"
]

MIN_GROUP_N = 20
SIGNIFICANCE_ALPHA = 0.05
GROUP_METHOD = "tertile"  # tertile, zero_vs_tertile, median
KG_MAX_HOPS = 4  # path-connected KG fallback, matching the mega KG idea
PAIRWISE_COMPARISONS = [("high", "low"), ("high", "mid"), ("mid", "low")]
EXPOSURE_SPECS


## Helper Functions

In [ ]:
def read_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".tsv", ".txt"}:
        return pd.read_csv(path, sep="\t")
    raise ValueError(f"Unsupported table: {path}")


def flatten_index(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if isinstance(out.index, pd.MultiIndex) or out.index.name is not None:
        out = out.reset_index()
    return out.loc[:, ~out.columns.duplicated()].copy()


def load_pheno_table(dataset: str, table: str, fallback_path: Path | None = None) -> pd.DataFrame:
    if fallback_path is not None and fallback_path.exists():
        return flatten_index(read_table(fallback_path))
    try:
        from pheno_utils import PhenoLoader
    except Exception as exc:
        raise RuntimeError(
            f"Could not import pheno_utils. Either run inside TRE or create {fallback_path}."
        ) from exc
    pl = PhenoLoader(dataset, age_sex_dataset=None, errors="warn")
    if table in pl.dfs:
        return flatten_index(pl.dfs[table])
    raise KeyError(f"Table {table!r} not in PhenoLoader({dataset!r}). Available: {list(pl.dfs)}")


def normalize_ids(df: pd.DataFrame, id_col: str = ID_COL) -> pd.DataFrame:
    out = flatten_index(df)
    if id_col not in out.columns:
        aliases = ["research_stage_id", "user_id", "RegistrationCode"]
        for alias in aliases:
            if alias in out.columns:
                out = out.rename(columns={alias: id_col})
                break
    if id_col not in out.columns:
        raise ValueError(f"Could not find participant id column in {out.columns.tolist()[:30]}")
    out[id_col] = out[id_col].astype(str)
    return out


def resolve_col(df: pd.DataFrame, requested: str, aliases: Iterable[str]) -> str:
    if requested in df.columns:
        return requested
    for alias in aliases:
        if alias in df.columns:
            return alias
    raise ValueError(f"Missing {requested}; tried aliases {list(aliases)}")


def normalize_token_text(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()


def token_contains(text: str, term: str) -> bool:
    clean_text = normalize_token_text(text)
    clean_term = normalize_token_text(term)
    if not clean_text or not clean_term:
        return False
    text_tokens = clean_text.split()
    term_tokens = clean_term.split()
    if len(term_tokens) == 1:
        return term_tokens[0] in text_tokens
    n = len(term_tokens)
    return any(text_tokens[i:i + n] == term_tokens for i in range(len(text_tokens) - n + 1))


def find_feature_column(food_features: pd.DataFrame, spec: dict) -> str | None:
    lower_to_col = {c.lower(): c for c in food_features.columns}
    for candidate in spec.get("preferred_feature_columns", []):
        if candidate in food_features.columns:
            return candidate
        if candidate.lower() in lower_to_col:
            return lower_to_col[candidate.lower()]

    terms = spec.get("terms", [])
    numeric_cols = list(food_features.select_dtypes(include="number").columns)
    matches = [c for c in numeric_cols if any(token_contains(c, t) for t in terms)]
    matches = [
        c for c in matches
        if not any(bad in c.lower() for bad in ["openfoodfacts_product", "product_processing_match"])
    ]
    return matches[0] if matches else None


def js_value_after_key(text: str, key: str):
    marker = f"{key}:"
    start = text.index(marker) + len(marker)
    decoder = json.JSONDecoder()
    value, _end = decoder.raw_decode(text[start:])
    return value


def load_mega_kg_data(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not path.exists():
        raise FileNotFoundError(f"Mega KG data file not found: {path}")
    text = path.read_text(encoding="utf-8")
    kinds = js_value_after_key(text, "kinds")
    relations = js_value_after_key(text, "relations")
    compact_nodes = js_value_after_key(text, "nodes")
    compact_edges = js_value_after_key(text, "edges")

    nodes = pd.DataFrame(compact_nodes, columns=["key", "label", "kind_idx", "x", "y"])
    nodes["kind"] = nodes["kind_idx"].map(lambda i: kinds[int(i)])
    nodes = nodes[["key", "kind", "label"]]

    key_by_idx = nodes["key"].astype(str).to_numpy()
    edges = pd.DataFrame(compact_edges, columns=["source_idx", "target_idx", "relation_idx"])
    edges["source"] = key_by_idx[edges["source_idx"].astype(int).to_numpy()]
    edges["target"] = key_by_idx[edges["target_idx"].astype(int).to_numpy()]
    edges["relation"] = edges["relation_idx"].map(lambda i: relations[int(i)])
    edges = edges[["source", "target", "relation"]]
    return nodes, edges


def load_kg_tables(mode: str = KG_MODE) -> tuple[pd.DataFrame, pd.DataFrame]:
    if mode == "full_mega":
        print("Loading full mega KG:", MEGA_KG_DATA_PATH)
        return load_mega_kg_data(MEGA_KG_DATA_PATH)
    if mode == "scenario":
        print("Loading scenario KG:", SCENARIO_KG_EDGES_PATH)
        nodes = read_table(SCENARIO_KG_NODES_PATH) if SCENARIO_KG_NODES_PATH.exists() else pd.DataFrame()
        edges = read_table(SCENARIO_KG_EDGES_PATH) if SCENARIO_KG_EDGES_PATH.exists() else pd.DataFrame()
        return nodes, edges
    raise ValueError(f"Unknown KG_MODE={mode!r}; use 'full_mega' or 'scenario'.")


def build_reverse_adjacency(kg_edges: pd.DataFrame) -> dict[str, set[str]]:
    reverse_adj: dict[str, set[str]] = {}
    if kg_edges is None or kg_edges.empty:
        return reverse_adj
    for source, target in kg_edges[["source", "target"]].dropna().astype(str).itertuples(index=False):
        reverse_adj.setdefault(target, set()).add(source)
    return reverse_adj


def kg_target_nodes_for_terms(kg_nodes: pd.DataFrame, kg_edges: pd.DataFrame, terms: list[str]) -> set[str]:
    target_nodes: set[str] = set()
    if kg_nodes is not None and not kg_nodes.empty:
        search_cols = [c for c in ["key", "label", "id"] if c in kg_nodes.columns]
        for row in kg_nodes[search_cols].fillna("").astype(str).itertuples(index=False, name=None):
            row_text = " | ".join(row)
            if any(token_contains(row_text, term) for term in terms):
                target_nodes.add(str(row[0]))
    if not target_nodes and kg_edges is not None and not kg_edges.empty:
        node_cols = [c for c in ["source", "target"] if c in kg_edges.columns]
        nodes = pd.unique(kg_edges[node_cols].astype(str).values.ravel("K"))
        target_nodes = {node for node in nodes if any(token_contains(node, term) for term in terms)}
    return target_nodes


def hpp_food_id_from_node(node: str) -> str | None:
    node = str(node)
    if node.startswith("hpp_food:"):
        return node.split(":", 1)[1]
    return None


def kg_path_connected_food_ids(
    kg_nodes: pd.DataFrame,
    kg_edges: pd.DataFrame,
    reverse_adj: dict[str, set[str]],
    terms: list[str],
    max_hops: int = KG_MAX_HOPS,
) -> tuple[set[str], dict]:
    """Find HPP foods connected to term-matching KG nodes within max_hops.

    The biologically relevant direction is food -> nutrients/chemicals/metabolites ->
    disease/pathway. To define exposure groups from a chemical/metabolite/pathway term,
    we start at matching target nodes and traverse reverse edges back toward hpp_food.
    This captures paths such as hpp_food -> canonical food -> FoodAtlas food -> chemical.
    """
    if kg_edges is None or kg_edges.empty:
        return set(), {"target_node_count": 0, "visited_node_count": 0, "max_hops": max_hops, "kg_mode": KG_MODE}

    target_nodes = kg_target_nodes_for_terms(kg_nodes, kg_edges, terms)
    frontier = set(target_nodes)
    visited = set(target_nodes)
    food_ids: set[str] = set()

    for depth in range(max_hops + 1):
        for node in frontier:
            food_id = hpp_food_id_from_node(node)
            if food_id is not None:
                food_ids.add(food_id)
        if depth == max_hops:
            break
        next_frontier = set()
        for node in frontier:
            next_frontier.update(reverse_adj.get(node, set()))
        next_frontier -= visited
        visited.update(next_frontier)
        frontier = next_frontier
        if not frontier:
            break

    example_targets = sorted(target_nodes)[:10]
    return food_ids, {
        "kg_mode": KG_MODE,
        "target_node_count": len(target_nodes),
        "visited_node_count": len(visited),
        "connected_food_count": len(food_ids),
        "max_hops": max_hops,
        "example_target_nodes": " | ".join(example_targets),
    }


def kg_connected_food_ids(kg_nodes: pd.DataFrame, kg_edges: pd.DataFrame, reverse_adj: dict[str, set[str]], terms: list[str]) -> set[str]:
    food_ids, _detail = kg_path_connected_food_ids(kg_nodes, kg_edges, reverse_adj, terms, max_hops=KG_MAX_HOPS)
    return food_ids


def participant_exposure_from_numeric_feature(
    diet_events: pd.DataFrame,
    food_features: pd.DataFrame,
    feature_col: str,
    exposure_name: str,
) -> pd.DataFrame:
    food_col = resolve_col(diet_events, FOOD_COL, ["food_id", "food_code", "item_id"])
    grams_col = resolve_col(diet_events, GRAMS_COL, ["weight_g", "amount_g", "grams", "serving_weight_g"])
    ref_col = REF_FOOD_COL if REF_FOOD_COL in food_features.columns else resolve_col(food_features, REF_FOOD_COL, ["food_id", "hpp_food_id"])

    events = diet_events[[ID_COL, food_col, grams_col]].copy()
    events[food_col] = events[food_col].astype(str)
    events[grams_col] = pd.to_numeric(events[grams_col], errors="coerce").fillna(0)

    features = food_features[[ref_col, feature_col]].copy()
    features[ref_col] = features[ref_col].astype(str)
    features[feature_col] = pd.to_numeric(features[feature_col], errors="coerce").fillna(0)

    merged = events.merge(features, left_on=food_col, right_on=ref_col, how="left")
    merged["exposure_value"] = merged[grams_col] / 100.0 * merged[feature_col].fillna(0)
    out = merged.groupby(ID_COL, as_index=False).agg(
        exposure_value=("exposure_value", "sum"),
        food_events=(food_col, "count"),
        unique_foods=(food_col, "nunique"),
    )
    out["exposure"] = exposure_name
    return out


def participant_exposure_from_kg_connected_foods(
    diet_events: pd.DataFrame,
    connected_food_ids: set[str],
    exposure_name: str,
) -> pd.DataFrame:
    food_col = resolve_col(diet_events, FOOD_COL, ["food_id", "food_code", "item_id"])
    grams_col = resolve_col(diet_events, GRAMS_COL, ["weight_g", "amount_g", "grams", "serving_weight_g"])
    d = diet_events[[ID_COL, food_col, grams_col]].copy()
    d[food_col] = d[food_col].astype(str)
    d[grams_col] = pd.to_numeric(d[grams_col], errors="coerce").fillna(0)
    d["is_connected_food"] = d[food_col].isin({str(x) for x in connected_food_ids})
    d["exposure_value"] = np.where(d["is_connected_food"], d[grams_col], 0.0)
    out = d.groupby(ID_COL, as_index=False).agg(
        exposure_value=("exposure_value", "sum"),
        connected_food_events=("is_connected_food", "sum"),
        food_events=(food_col, "count"),
        unique_foods=(food_col, "nunique"),
    )
    out["exposure"] = exposure_name
    return out


def assign_exposure_groups(values: pd.Series, method: str = GROUP_METHOD) -> pd.Series:
    v = pd.to_numeric(values, errors="coerce").fillna(0)
    if method == "zero_vs_tertile":
        out = pd.Series("low", index=v.index, dtype="object")
        positive = v[v > 0]
        if positive.nunique() >= 3:
            out.loc[positive.index] = pd.qcut(positive, 3, labels=["low_positive", "mid", "high"], duplicates="drop").astype(str)
            out = out.replace({"low_positive": "low"})
        elif len(positive):
            out.loc[positive.index] = "high"
        return out
    if method == "median":
        med = v[v > 0].median()
        return pd.Series(np.where(v > med, "high", "low"), index=v.index)
    if v.nunique() >= 3:
        return pd.qcut(v.rank(method="first"), 3, labels=["low", "mid", "high"]).astype(str)
    return pd.Series(np.where(v > 0, "high", "low"), index=v.index)


In [ ]:
def select_mental_health_columns(df: pd.DataFrame, patterns: list[str] = DEFAULT_OUTCOME_PATTERNS) -> list[str]:
    cols = []
    for c in df.columns:
        cl = c.lower()
        if any(p in cl for p in patterns):
            vals = pd.to_numeric(df[c], errors="coerce")
            if vals.notna().sum() >= MIN_GROUP_N:
                cols.append(c)
    return cols


def add_composite_scores(mental: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    out = mental.copy()
    depression_cols = [c for c in out.columns if any(p in c.lower() for p in ["depress", "disinterest"])]
    anxiety_cols = [c for c in out.columns if any(p in c.lower() for p in ["nervous", "worrier", "worry", "tense", "restless", "fed_up"])]
    composites = []
    for name, cols in [("composite_depression", depression_cols), ("composite_anxiety_worry", anxiety_cols)]:
        numeric = []
        for c in cols:
            vals = pd.to_numeric(out[c], errors="coerce")
            if vals.notna().sum() >= MIN_GROUP_N:
                out[c] = vals
                numeric.append(c)
        if len(numeric) >= 2:
            z = out[numeric].apply(pd.to_numeric, errors="coerce")
            z = (z - z.mean()) / z.std(ddof=0).replace(0, np.nan)
            out[name] = z.mean(axis=1)
            composites.append(name)
    return out, composites


def finite_numeric_values(series: pd.Series) -> np.ndarray:
    vals = pd.to_numeric(series, errors="coerce")
    vals = vals.replace([np.inf, -np.inf], np.nan).dropna()
    return vals.astype(float).to_numpy()


def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) == 0 or len(y) == 0:
        return np.nan
    greater = 0
    less = 0
    for value in x:
        greater += np.sum(value > y)
        less += np.sum(value < y)
    return float((greater - less) / (len(x) * len(y)))


def significance_label(p_value: float) -> str:
    return "yes" if pd.notna(p_value) and p_value < SIGNIFICANCE_ALPHA else "no"


def pairwise_tests(df: pd.DataFrame, exposure_name: str, outcome_cols: list[str]) -> pd.DataFrame:
    rows = []
    for outcome in outcome_cols:
        for a, b in PAIRWISE_COMPARISONS:
            xa = finite_numeric_values(df.loc[df["exposure_group"] == a, outcome])
            xb = finite_numeric_values(df.loc[df["exposure_group"] == b, outcome])
            if len(xa) < MIN_GROUP_N or len(xb) < MIN_GROUP_N:
                rows.append({
                    "exposure": exposure_name,
                    "outcome": outcome,
                    "comparison": f"{a}_vs_{b}",
                    "group_a": a,
                    "group_b": b,
                    "n_a": len(xa),
                    "n_b": len(xb),
                    "median_a": np.nan,
                    "median_b": np.nan,
                    "effect_median_diff": np.nan,
                    "effect_cliffs_delta": np.nan,
                    "effect_direction": "not_tested",
                    "p_value": np.nan,
                    "significant": "no",
                    "test": "mannwhitneyu",
                    "note": "too_few_samples",
                })
                continue

            try:
                res = stats.mannwhitneyu(xa, xb, alternative="two-sided")
                p_value = float(res.pvalue)
                note = "ok"
            except ValueError as exc:
                p_value = np.nan
                note = str(exc)

            median_diff = float(np.median(xa) - np.median(xb))
            delta = cliffs_delta(xa, xb)
            if median_diff > 0:
                direction = f"{a}_higher"
            elif median_diff < 0:
                direction = f"{a}_lower"
            else:
                direction = "no_median_difference"

            rows.append({
                "exposure": exposure_name,
                "outcome": outcome,
                "comparison": f"{a}_vs_{b}",
                "group_a": a,
                "group_b": b,
                "n_a": len(xa),
                "n_b": len(xb),
                "median_a": float(np.median(xa)),
                "median_b": float(np.median(xb)),
                "effect_median_diff": median_diff,
                "effect_cliffs_delta": delta,
                "effect_direction": direction,
                "p_value": p_value,
                "significant": significance_label(p_value),
                "test": "mannwhitneyu",
                "note": note,
            })
    result = pd.DataFrame(rows)
    if result.empty:
        return result
    return result.sort_values("p_value", na_position="last")


def overall_tests(df: pd.DataFrame, exposure_name: str, outcome_cols: list[str]) -> pd.DataFrame:
    rows = []
    for outcome in outcome_cols:
        groups = []
        group_names = []
        for g in ["low", "mid", "high"]:
            vals = finite_numeric_values(df.loc[df["exposure_group"] == g, outcome])
            if len(vals) >= MIN_GROUP_N:
                groups.append(vals)
                group_names.append(g)
        if len(groups) < 2:
            rows.append({
                "exposure": exposure_name,
                "outcome": outcome,
                "test": "kruskal",
                "groups_used": ",".join(group_names),
                "p_value": np.nan,
                "significant": "no",
                "statistic": np.nan,
                "note": "too_few_groups",
            })
            continue
        try:
            stat, p = stats.kruskal(*groups)
            p_value = float(p)
            rows.append({
                "exposure": exposure_name,
                "outcome": outcome,
                "test": "kruskal",
                "groups_used": ",".join(group_names),
                "p_value": p_value,
                "significant": significance_label(p_value),
                "statistic": float(stat),
                "note": "ok",
            })
        except ValueError as exc:
            rows.append({
                "exposure": exposure_name,
                "outcome": outcome,
                "test": "kruskal",
                "groups_used": ",".join(group_names),
                "p_value": np.nan,
                "significant": "no",
                "statistic": np.nan,
                "note": str(exc),
            })
    result = pd.DataFrame(rows)
    if result.empty:
        return result
    return result.sort_values("p_value", na_position="last")


def plot_exposure_outcome_boxplot(df: pd.DataFrame, exposure_name: str, outcome: str, stats_df: pd.DataFrame | None = None, save: bool = True):
    order = ["low", "mid", "high"]
    plot_df = df[df["exposure_group"].isin(order)].copy()
    plot_df[outcome] = pd.to_numeric(plot_df[outcome], errors="coerce").replace([np.inf, -np.inf], np.nan)
    fig, ax = plt.subplots(figsize=(7.5, 5))
    sns.boxplot(data=plot_df, x="exposure_group", y=outcome, order=order, ax=ax, color="#d9e6f2")
    sns.stripplot(data=plot_df, x="exposure_group", y=outcome, order=order, ax=ax, color="#19324a", alpha=0.25, size=2)
    ax.set_title(f"{outcome}\nby {exposure_name} exposure group")
    ax.set_xlabel(f"{exposure_name} exposure group")
    ax.set_ylabel(outcome)
    counts = plot_df.groupby("exposure_group")[outcome].apply(lambda s: s.notna().sum()).reindex(order)
    ax.set_xticklabels([f"{g}\nn={int(counts.get(g, 0) or 0)}" for g in order])
    if stats_df is not None and not stats_df.empty:
        sub = stats_df[(stats_df["exposure"] == exposure_name) & (stats_df["outcome"] == outcome)]
        lines = []
        for comp in ["high_vs_low", "high_vs_mid", "mid_vs_low"]:
            hit = sub[sub["comparison"] == comp]
            if not hit.empty and pd.notna(hit.iloc[0]["p_value"]):
                lines.append(
                    f"{comp}: p={hit.iloc[0]['p_value']:.2e}, delta={hit.iloc[0]['effect_cliffs_delta']:.3f}"
                )
        if lines:
            ax.text(0.02, 0.98, "\n".join(lines), transform=ax.transAxes, va="top", ha="left", fontsize=9,
                    bbox={"boxstyle": "round", "facecolor": "white", "edgecolor": "#b9c6d3", "alpha": 0.9})
    fig.tight_layout()
    if save:
        safe_outcome = re.sub(r"[^A-Za-z0-9_.-]+", "_", outcome)
        path = OUTPUT_DIR / f"boxplot_{exposure_name}_{safe_outcome}.png"
        fig.savefig(path, dpi=160)
        print(f"Saved {path}")
    return fig, ax


def plot_effect_size_forest(
    pairwise_results: pd.DataFrame,
    exposure: str | None = None,
    comparison: str = "high_vs_low",
    significant_only: bool = False,
    top_n: int = 25,
    save: bool = True,
):
    df = pairwise_results.copy()
    if exposure is not None:
        df = df[df["exposure"] == exposure]
    df = df[df["comparison"] == comparison]
    df = df[df["effect_cliffs_delta"].notna()]
    if significant_only and "significant" in df.columns:
        df = df[df["significant"] == "yes"]
    df = df.sort_values("effect_cliffs_delta", key=lambda s: s.abs(), ascending=False).head(top_n)
    df = df.sort_values("effect_cliffs_delta")
    if df.empty:
        print("No effects to plot.")
        return None, None
    fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * len(df))))
    colors = np.where(df["significant"].eq("yes"), "#b91c1c", "#64748b")
    ax.scatter(df["effect_cliffs_delta"], df["outcome"], c=colors, s=45)
    ax.axvline(0, color="#111827", linewidth=1)
    for _, row in df.iterrows():
        label = f"p={row['p_value']:.2e}" if pd.notna(row["p_value"]) else "p=NA"
        ax.text(row["effect_cliffs_delta"], row["outcome"], "  " + label, va="center", fontsize=8)
    ax.set_xlabel(f"Cliff's delta ({comparison})")
    ax.set_ylabel("Mental-health outcome")
    ax.set_title(f"Effect-size forest plot: {exposure or 'all exposures'}")
    fig.tight_layout()
    if save:
        safe_exposure = re.sub(r"[^A-Za-z0-9_.-]+", "_", exposure or "all_exposures")
        path = OUTPUT_DIR / f"forest_effects_{safe_exposure}_{comparison}.png"
        fig.savefig(path, dpi=180)
        print(f"Saved {path}")
    return fig, ax


## Load Diet, Food Feature/KG, And Mental Health Data

In [ ]:
# Load diet events. Prefer existing tre_inputs cache, but if it is missing or unreadable,
# read the official HPP diet_logging_events parquet directly as shown in the Pheno notebook.
try:
    diet_events = read_table(DIET_EVENTS_PATH)
    print("Loaded cached diet events:", DIET_EVENTS_PATH, diet_events.shape)
except Exception as cache_error:
    print(f"Could not read cached diet events at {DIET_EVENTS_PATH}: {cache_error}")
    print("Trying direct PhenoLoader/HPP dataset export...")

    from pheno_utils import PhenoLoader
    from pheno_utils.config import DATASETS_PATH

    tre_inputs = PROJECT_ROOT / "tre_inputs"
    tre_inputs.mkdir(parents=True, exist_ok=True)

    pl = PhenoLoader("diet_logging", age_sex_dataset=None, errors="warn")
    dataset_dir = Path(DATASETS_PATH) / pl.dataset

    candidate_paths = [dataset_dir / "diet_logging_events.parquet"]

    if "diet_logging" in pl.dfs:
        summary_df = pl.dfs["diet_logging"].reset_index()
        if "diet_logging_events" in summary_df.columns:
            for rel_path in summary_df["diet_logging_events"].dropna().astype(str).unique()[:10]:
                candidate_paths.append(dataset_dir / rel_path)

    events_path = next((path for path in candidate_paths if path.exists()), None)
    if events_path is None:
        raise FileNotFoundError(
            "Could not find diet_logging_events.parquet. Tried:\n"
            + "\n".join(str(path) for path in candidate_paths)
        )

    print("Reading official diet event parquet:", events_path)
    diet_events = pd.read_parquet(events_path).reset_index()
    diet_events = diet_events.loc[:, ~diet_events.columns.duplicated()]
    print("Loaded diet events from source:", diet_events.shape)

    try:
        diet_events.to_parquet(DIET_EVENTS_PATH, index=False)
        print("Cached diet events for next run:", DIET_EVENTS_PATH)
    except Exception as write_error:
        print("Could not cache diet events; continuing with in-memory table:", write_error)

food_features = read_table(FOOD_FEATURE_TABLE)
kg_nodes, kg_edges = load_kg_tables(KG_MODE)
kg_reverse_adj = build_reverse_adjacency(kg_edges)
mental = load_pheno_table(MENTAL_HEALTH_DATASET, MENTAL_HEALTH_TABLE, MENTAL_HEALTH_LOCAL_PATH)

diet_events = normalize_ids(diet_events)
mental = normalize_ids(mental)
mental, composite_cols = add_composite_scores(mental)
mental_outcome_cols = select_mental_health_columns(mental) + composite_cols
mental_outcome_cols = list(dict.fromkeys(mental_outcome_cols))

print("diet_events", diet_events.shape)
print("food_features", food_features.shape)
print("kg_mode", KG_MODE)
print("kg_nodes", kg_nodes.shape)
print("kg_edges", kg_edges.shape)
print("mental", mental.shape)
print("mental outcome columns", len(mental_outcome_cols))
mental_outcome_cols[:30]


## Build Participant Exposure Groups

For each chemical/exposure:

1. Prefer a numeric food-level feature column, such as `Caffeine` per 100 g.
2. If no numeric feature exists, use KG connected foods and sum grams of connected foods.
3. If neither exists, write a warning and skip that exposure.

In [ ]:
def build_one_exposure(exposure_name: str, spec: dict) -> tuple[pd.DataFrame | None, dict]:
    feature_col = find_feature_column(food_features, spec)
    if feature_col:
        exposure = participant_exposure_from_numeric_feature(diet_events, food_features, feature_col, exposure_name)
        detail = {
            "exposure": exposure_name,
            "source": f"numeric_feature:{feature_col}",
            "connected_food_count": None,
            "kg_mode": KG_MODE,
            "note": "numeric feature source; KG connected count not used",
        }
    else:
        connected, kg_detail = kg_path_connected_food_ids(
            kg_nodes,
            kg_edges,
            kg_reverse_adj,
            spec.get("terms", [exposure_name]),
            max_hops=KG_MAX_HOPS,
        )
        if not connected:
            detail = {
                "exposure": exposure_name,
                "source": "not_found",
                "connected_food_count": 0,
                **kg_detail,
                "note": spec.get("source_note", ""),
            }
            return None, detail
        exposure = participant_exposure_from_kg_connected_foods(diet_events, connected, exposure_name)
        detail = {
            "exposure": exposure_name,
            "source": "kg_path_connected_food_grams",
            "connected_food_count": len(connected),
            **kg_detail,
        }
    exposure["exposure_group"] = assign_exposure_groups(exposure["exposure_value"], method=GROUP_METHOD)
    detail["participants"] = int(exposure[ID_COL].nunique())
    detail["nonzero_participants"] = int((exposure["exposure_value"] > 0).sum())
    detail["group_counts"] = exposure["exposure_group"].value_counts(dropna=False).to_dict()
    return exposure, detail


exposure_tables = {}
exposure_summaries = []
for exposure_name, spec in EXPOSURE_SPECS.items():
    exposure, detail = build_one_exposure(exposure_name, spec)
    exposure_summaries.append(detail)
    if exposure is not None:
        exposure_tables[exposure_name] = exposure

exposure_summary = pd.DataFrame(exposure_summaries)
exposure_summary.to_csv(OUTPUT_DIR / "exposure_build_summary.csv", index=False)
exposure_summary


## Run Statistical Comparisons

In [ ]:
all_pairwise = []
all_overall = []
merged_by_exposure = {}

for exposure_name, exposure in exposure_tables.items():
    merged = exposure.merge(mental[[ID_COL] + mental_outcome_cols], on=ID_COL, how="inner")
    merged_by_exposure[exposure_name] = merged
    pairwise = pairwise_tests(merged, exposure_name, mental_outcome_cols)
    overall = overall_tests(merged, exposure_name, mental_outcome_cols)
    all_pairwise.append(pairwise)
    all_overall.append(overall)

pairwise_results = pd.concat(all_pairwise, ignore_index=True) if all_pairwise else pd.DataFrame()
overall_results = pd.concat(all_overall, ignore_index=True) if all_overall else pd.DataFrame()
pairwise_results.to_csv(OUTPUT_DIR / "mental_health_pairwise_results.csv", index=False)
overall_results.to_csv(OUTPUT_DIR / "mental_health_overall_results.csv", index=False)

pairwise_results.head(20)

## Heatmap: Which Exposure Differs For Which Mental-Health Metric?

This gives a compact side-paper figure: rows are food chemicals/exposures, columns are mental-health metrics, and cells show `-log10(q)` for high-vs-low comparisons.

In [ ]:
def plot_significance_heatmap(results: pd.DataFrame, comparison: str = "high_vs_low"):
    sub = results[(results["comparison"] == comparison) & results["p_value"].notna()].copy()
    if sub.empty:
        print("No valid pairwise results to plot.")
        return None
    sub["neg_log10_p"] = -np.log10(sub["p_value"].clip(lower=1e-300))
    pivot = sub.pivot_table(index="exposure", columns="outcome", values="neg_log10_p", aggfunc="max")
    keep_cols = pivot.max(axis=0).sort_values(ascending=False).head(25).index
    pivot = pivot[keep_cols]
    fig, ax = plt.subplots(figsize=(min(18, 1 + 0.55 * len(pivot.columns)), 1.5 + 0.6 * len(pivot)))
    sns.heatmap(pivot, cmap="viridis", linewidths=0.4, linecolor="white", ax=ax, cbar_kws={"label": "-log10(raw p)"})
    ax.set_title(f"Mental-health associations by diet exposure ({comparison})")
    ax.set_xlabel("mental-health metric")
    ax.set_ylabel("diet-derived exposure")
    fig.tight_layout()
    path = OUTPUT_DIR / f"heatmap_{comparison}.png"
    fig.savefig(path, dpi=180)
    print(f"Saved {path}")
    return fig, ax

plot_significance_heatmap(pairwise_results, comparison="high_vs_low")
plot_effect_size_forest(pairwise_results, comparison="high_vs_low", significant_only=False, top_n=25)


## Plot A Selected Exposure And Outcome

Change `SELECTED_EXPOSURE` and `SELECTED_OUTCOME` to quickly regenerate the boxplot with p-values. If `SELECTED_OUTCOME = None`, the notebook chooses the strongest high-vs-low result for that exposure.

In [ ]:
SELECTED_EXPOSURE = "caffeine"
SELECTED_OUTCOME = None  # e.g. "health_mental_past_two_week_depression_frequency" or "composite_depression"

if SELECTED_EXPOSURE not in merged_by_exposure:
    print(f"Exposure {SELECTED_EXPOSURE!r} was not available. Available: {list(merged_by_exposure)}")
else:
    if SELECTED_OUTCOME is None:
        candidates = pairwise_results[(pairwise_results["exposure"] == SELECTED_EXPOSURE) & (pairwise_results["comparison"] == "high_vs_low")]
        candidates = candidates[candidates["p_value"].notna()].sort_values("p_value")
        SELECTED_OUTCOME = candidates.iloc[0]["outcome"] if not candidates.empty else mental_outcome_cols[0]
    plot_exposure_outcome_boxplot(
        merged_by_exposure[SELECTED_EXPOSURE],
        SELECTED_EXPOSURE,
        SELECTED_OUTCOME,
        stats_df=pairwise_results,
    )
    plot_effect_size_forest(
        pairwise_results,
        exposure=SELECTED_EXPOSURE,
        comparison="high_vs_low",
        significant_only=False,
        top_n=25,
    )


## Sex-Stratified Depression Burden Plots

Create four publication-style panels/files: male and female strata for longest depression duration and depression episode count, with raw p-values for the selected exposure comparison.


In [ ]:
SEX_STRATIFIED_EXPOSURE = SELECTED_EXPOSURE
SEX_COMPARISON = ("high", "low")

# Set these manually if the automatic finder picks the wrong TRE columns.
SEX_COL = None
LONGEST_DEPRESSION_TIME_COL = None
DEPRESSION_EPISODE_COUNT_COL = None


def find_first_column(df: pd.DataFrame, patterns: list[str], numeric: bool | None = None) -> str | None:
    for col in df.columns:
        label = normalize_token_text(col)
        if all(p in label for p in [normalize_token_text(x) for x in patterns]):
            if numeric is True and pd.to_numeric(df[col], errors="coerce").notna().sum() < MIN_GROUP_N:
                continue
            return col
    return None


def infer_sex_column(df: pd.DataFrame) -> str | None:
    candidates = ["sex", "gender", "biological_sex"]
    lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c in lower:
            return lower[c]
    for col in df.columns:
        label = normalize_token_text(col)
        if label in {"sex", "gender", "biological sex"}:
            return col
    return None


def normalize_sex_values(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()
    mapping = {
        "m": "male", "male": "male", "1": "male", "man": "male",
        "f": "female", "female": "female", "0": "female", "woman": "female",
    }
    return s.map(mapping).fillna(s)


def resolve_depression_burden_columns(mental: pd.DataFrame) -> tuple[str | None, str | None, str | None]:
    sex_col = SEX_COL or infer_sex_column(mental)
    longest_col = LONGEST_DEPRESSION_TIME_COL
    episode_col = DEPRESSION_EPISODE_COUNT_COL

    if longest_col is None:
        longest_col = (
            find_first_column(mental, ["depress", "longest"], numeric=True)
            or find_first_column(mental, ["depression", "duration"], numeric=True)
            or find_first_column(mental, ["depress", "time"], numeric=True)
        )
    if episode_col is None:
        episode_col = (
            find_first_column(mental, ["depress", "episode"], numeric=True)
            or find_first_column(mental, ["depression", "episodes"], numeric=True)
            or find_first_column(mental, ["depress", "number"], numeric=True)
        )
    return sex_col, longest_col, episode_col


def p_value_for_groups(df: pd.DataFrame, outcome: str, group_a: str, group_b: str) -> tuple[float, float, int, int]:
    xa = finite_numeric_values(df.loc[df["exposure_group"] == group_a, outcome])
    xb = finite_numeric_values(df.loc[df["exposure_group"] == group_b, outcome])
    if len(xa) < MIN_GROUP_N or len(xb) < MIN_GROUP_N:
        return np.nan, np.nan, len(xa), len(xb)
    try:
        p = float(stats.mannwhitneyu(xa, xb, alternative="two-sided").pvalue)
    except ValueError:
        p = np.nan
    return p, cliffs_delta(xa, xb), len(xa), len(xb)


def plot_one_sex_depression_panel(
    df: pd.DataFrame,
    sex_value: str,
    outcome: str,
    outcome_label: str,
    exposure_name: str,
    comparison: tuple[str, str] = SEX_COMPARISON,
    ax=None,
    save_single: bool = True,
):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6.8, 5.0))
    else:
        fig = ax.figure

    group_a, group_b = comparison
    order = ["low", "mid", "high"]
    plot_df = df[(df["sex_normalized"] == sex_value) & (df["exposure_group"].isin(order))].copy()
    plot_df[outcome] = pd.to_numeric(plot_df[outcome], errors="coerce").replace([np.inf, -np.inf], np.nan)

    sns.boxplot(data=plot_df, x="exposure_group", y=outcome, order=order, ax=ax, color="#e6eef6")
    sns.stripplot(data=plot_df, x="exposure_group", y=outcome, order=order, ax=ax, color="#1f2937", alpha=0.25, size=2)

    p, delta, n_a, n_b = p_value_for_groups(plot_df, outcome, group_a, group_b)
    p_text = "p=NA" if pd.isna(p) else f"p={p:.2e}"
    delta_text = "delta=NA" if pd.isna(delta) else f"delta={delta:.3f}"

    counts = plot_df.groupby("exposure_group")[outcome].apply(lambda s: s.notna().sum()).reindex(order)
    ax.set_xticklabels([f"{g}\nn={int(counts.get(g, 0) or 0)}" for g in order])
    ax.set_title(f"{sex_value.title()}: {outcome_label}")
    ax.set_xlabel(f"{exposure_name} exposure group")
    ax.set_ylabel(outcome_label)
    ax.text(
        0.02, 0.98,
        f"{group_a} vs {group_b}: {p_text}\nCliff's {delta_text}\nn={group_a}:{n_a}, {group_b}:{n_b}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=9,
        bbox={"boxstyle": "round", "facecolor": "white", "edgecolor": "#b9c6d3", "alpha": 0.92},
    )

    if save_single:
        safe_outcome = re.sub(r"[^A-Za-z0-9_.-]+", "_", outcome_label.lower())
        path = OUTPUT_DIR / f"sex_stratified_{exposure_name}_{sex_value}_{safe_outcome}.png"
        fig.tight_layout()
        fig.savefig(path, dpi=180)
        print(f"Saved {path}")
    return fig, ax


if SEX_STRATIFIED_EXPOSURE not in merged_by_exposure:
    print(f"Exposure {SEX_STRATIFIED_EXPOSURE!r} was not available. Available: {list(merged_by_exposure)}")
else:
    sex_col, longest_col, episode_col = resolve_depression_burden_columns(mental)
    print("sex_col:", sex_col)
    print("longest depression time col:", longest_col)
    print("depression episode count col:", episode_col)

    missing = [name for name, col in {
        "sex": sex_col,
        "longest_depression_time": longest_col,
        "depression_episode_count": episode_col,
    }.items() if col is None]

    if missing:
        print("Missing columns:", missing)
        print("Candidate mental-health columns containing depression/depress:")
        display(pd.DataFrame({"column": [c for c in mental.columns if "depress" in c.lower() or "depression" in c.lower()]}).head(80))
    else:
        sex_table = mental[[ID_COL, sex_col, longest_col, episode_col]].copy()
        sex_table["sex_normalized"] = normalize_sex_values(sex_table[sex_col])
        plot_data = merged_by_exposure[SEX_STRATIFIED_EXPOSURE].merge(sex_table, on=ID_COL, how="inner")

        fig, axes = plt.subplots(2, 2, figsize=(13.5, 9.5), sharex=False)
        plot_one_sex_depression_panel(plot_data, "male", longest_col, "Longest depression time", SEX_STRATIFIED_EXPOSURE, ax=axes[0, 0], save_single=True)
        plot_one_sex_depression_panel(plot_data, "female", longest_col, "Longest depression time", SEX_STRATIFIED_EXPOSURE, ax=axes[0, 1], save_single=True)
        plot_one_sex_depression_panel(plot_data, "male", episode_col, "Depression episode count", SEX_STRATIFIED_EXPOSURE, ax=axes[1, 0], save_single=True)
        plot_one_sex_depression_panel(plot_data, "female", episode_col, "Depression episode count", SEX_STRATIFIED_EXPOSURE, ax=axes[1, 1], save_single=True)
        fig.suptitle(f"Sex-stratified depression burden by {SEX_STRATIFIED_EXPOSURE} exposure", y=1.02)
        fig.tight_layout()
        combined_path = OUTPUT_DIR / f"sex_stratified_{SEX_STRATIFIED_EXPOSURE}_depression_burden_2x2.png"
        fig.savefig(combined_path, dpi=180, bbox_inches="tight")
        print(f"Saved {combined_path}")


## Save Analysis Tables

In [ ]:
for exposure_name, merged in merged_by_exposure.items():
    merged.to_parquet(OUTPUT_DIR / f"participant_exposure_mental_health_{exposure_name}.parquet", index=False)

summary = {
    "exposures_requested": list(EXPOSURE_SPECS),
    "exposures_available": list(merged_by_exposure),
    "mental_health_dataset": MENTAL_HEALTH_DATASET,
    "mental_health_table": MENTAL_HEALTH_TABLE,
    "mental_outcome_count": len(mental_outcome_cols),
    "output_dir": str(OUTPUT_DIR),
    "notes": "Exploratory association only; not adjusted for confounders by default.",
}
(OUTPUT_DIR / "diet_mental_health_analysis_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary

## Suggested Next Steps

- Add covariate adjustment: age, sex, BMI, socioeconomic status, smoking, alcohol, total energy intake, and medication/diagnosis variables.
- Replace simple tertiles with domain-specific thresholds when external toxicology/nutrition thresholds are defensible.
- Treat lead/acrylamide as KG-connected or food-processing exposure hypotheses only if the KG/reference table actually contains those annotations.
- Run a sensitivity analysis using NutriMatch-based features and the weighted KG-enhanced graph.